# Capítulo 12 — Ciência de dados: analisando a missão

## Missão
Aplicar o processo de ciência de dados aos dados da missão:
1. capturar;
2. limpar;
3. explorar;
4. modelar;
5. comunicar.

Com NumPy, pandas e Matplotlib, você analisa um mês simulado de telemetria. Depois troca o BigQuery do livro (que exige conta no Google Cloud) por um conjunto grande gerado localmente: **1 milhão de linhas**, agregadas em segundos.

**No fim deste caderno você terá:** o módulo `gabarito_rover/analise.py`, que gera, limpa e resume a telemetria, e um relatório com gráficos da missão.

**Leitura essencial (opcional, livro em inglês):**
- Cooking with Gas: The Five-Step Process of Data Science · PDF p. 456
- Introducing NumPy, Pandas, and MatPlotLib · PDF p. 460
- Doing Your First Data Science Project · PDF p. 462
- Reading the Medicare Big Data · PDF p. 478

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
- **NumPy** faz contas em vetores inteiros, sem laço.
- **pandas** é uma planilha programável (`DataFrame`): filtrar, agrupar, juntar, limpar.
- **Matplotlib** desenha.

O segredo para dados grandes é deixar o pandas/NumPy fazer o trabalho em bloco (vetorizado) em vez de usar `for` linha a linha.

### 12.1 **Gerador de telemetria.** Escreva `analise.py` com `gerar_telemetria(dias=30, semente=0)`. Ela devolve um `DataFrame` com uma linha por hora, com:
- `hora` (`pd.date_range` a partir de 2026-03-01, em UTC);
- `temperatura`: um ciclo diário com seno (mínima ~-75 às 5h, máxima ~-10 às 14h) mais ruído;
- `poeira` (entre 0 e 1);
- `bateria`: cai de dia e recarrega até 100;
- `distancia` (0–400).

Para testar a limpeza, coloque **de propósito** 2% de temperaturas faltando e 5 leituras impossíveis (+500 °C).

_A célula grava `gabarito_rover/analise.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/analise.py
"""Geração, limpeza e resumo da telemetria do rover."""
import numpy as np
import pandas as pd


def gerar_telemetria(dias: int = 30, semente: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(semente)
    hora = pd.date_range("2026-03-01", periods=dias * 24, freq="h", tz="UTC")
    h = hora.hour.to_numpy()
    temperatura = -42.5 + 32.5 * np.sin((h - 9.5) / 24 * 2 * np.pi) + rng.normal(0, 3, len(h))
    poeira = np.clip(rng.beta(2, 5, len(h)) + 0.15 * (h >= 11) * (h <= 15), 0, 1)
    bateria = np.empty(len(h))
    nivel = 100.0
    for i, hh in enumerate(h):
        nivel = min(100.0, nivel + 6) if (hh < 6 or hh > 20) else max(5.0, nivel - rng.uniform(1, 4))
        bateria[i] = nivel
    distancia = rng.integers(0, 401, len(h))
    df = pd.DataFrame({"hora": hora, "temperatura": temperatura.round(1), "poeira": poeira.round(2),
                       "bateria": bateria.round(1), "distancia": distancia})
    faltando = rng.choice(len(df), int(0.02 * len(df)), replace=False)
    df.loc[faltando, "temperatura"] = np.nan
    df.loc[rng.choice(len(df), 5, replace=False), "temperatura"] = 500.0
    return df


def limpar(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.loc[~df.temperatura.between(-130, 40), "temperatura"] = np.nan    # impossíveis viram faltantes
    df["temperatura"] = df.temperatura.interpolate(limit_direction="both")
    return df


def resumo_diario(df: pd.DataFrame) -> pd.DataFrame:
    return (df.set_index("hora").resample("D")
              .agg(temp_min=("temperatura", "min"), temp_max=("temperatura", "max"),
                   poeira_media=("poeira", "mean"), bateria_min=("bateria", "min")).round(2))


### 12.2 **Capturar e inspecionar.** Importe `numpy`, `pandas` e `matplotlib.pyplot` (como `np`, `pd` e `plt`), gere 30 dias e mostre: `shape`, `head()`, `info()` e `describe()`. Quantos valores faltam em cada coluna? Há algo impossível nas temperaturas?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gabarito_rover.analise import gerar_telemetria, limpar, resumo_diario
df = gerar_telemetria(30)
print(df.shape)
display(df.head())
df.info()
display(df.describe().round(1))
print(df.isna().sum())

### 12.3 **NumPy vetorizado.** Converta todas as temperaturas para Fahrenheit de dois jeitos: com um `for` em Python e com NumPy direto no array. Compare o tempo com `time.perf_counter` (use 1 milhão de valores aleatórios).

In [ ]:
arr = np.random.default_rng(0).normal(-40, 20, 1_000_000)
t0 = time.perf_counter(); lista = [c * 9 / 5 + 32 for c in arr]; t_for = time.perf_counter() - t0
t0 = time.perf_counter(); vet = arr * 9 / 5 + 32; t_np = time.perf_counter() - t0
print(f"for: {t_for:.3f}s | NumPy: {t_np:.4f}s | {t_for / t_np:.0f}x mais rápido | iguais: {np.allclose(lista, vet)}")

### 12.4 **Limpar.** Use `limpar(df)` e confira: nenhuma temperatura fora de -130 a 40 e nenhum valor faltando. Mostre as linhas que tinham 500 °C antes e depois.

In [ ]:
sujas = df.index[df.temperatura > 40]
limpo = limpar(df)
print(limpo.temperatura.between(-130, 40).all(), limpo.temperatura.isna().sum())
pd.DataFrame({"antes": df.loc[sujas, "temperatura"], "depois": limpo.loc[sujas, "temperatura"]})

### 12.5 **Explorar.** Calcule a temperatura média por **hora do dia** (`groupby(df.hora.dt.hour)`) e desenhe o ciclo diário. Em que hora a poeira é maior? Mostre a correlação entre as colunas numéricas.

In [ ]:
por_hora = limpo.groupby(limpo.hora.dt.hour)[["temperatura", "poeira"]].mean()
ax = por_hora.temperatura.plot(marker="o", title="Ciclo diário da temperatura")
ax.set_xlabel("hora (UTC)"); ax.set_ylabel("°C"); plt.show()
print("hora com mais poeira:", por_hora.poeira.idxmax())
limpo[["temperatura", "poeira", "bateria", "distancia"]].corr().round(2)

### 12.6 **Resumo diário e gráfico de comunicação.** Use `resumo_diario` e faça uma figura com 2 gráficos: as faixas mínima e máxima de temperatura por dia (`fill_between`) e a bateria mínima por dia com uma linha de alerta em 20%.

In [ ]:
rd = resumo_diario(limpo)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a1.fill_between(rd.index, rd.temp_min, rd.temp_max, alpha=0.4); a1.set_ylabel("°C"); a1.set_title("Temperatura diária")
a2.plot(rd.index, rd.bateria_min, marker="."); a2.axhline(20, color="red", ls="--"); a2.set_ylabel("bateria mín. %")
plt.tight_layout(); plt.show()
rd.head()

### 12.7 **Modelar (simples).** Ajuste uma reta (`np.polyfit`, grau 1) entre poeira e bateria e diga quanto a bateria cai, em média, para cada +0,1 de poeira.

In [ ]:
a, b = np.polyfit(limpo.poeira, limpo.bateria, 1)
print(f"bateria ≈ {a:.1f} × poeira + {b:.1f} -> a cada +0,1 de poeira: {a * 0.1:+.2f} pontos")

### 12.8 **Dados grandes sem nuvem.** Gere 1 milhão de leituras de 20 rovers fictícios (colunas `rover`, `temperatura`, `poeira`, com NumPy direto), e com pandas encontre o rover com a maior poeira média e os 3 mais frios. Grave tudo em CSV, mostre o tamanho do arquivo em MB e meça o tempo total.

In [ ]:
t0 = time.perf_counter()
rng = np.random.default_rng(42)
n = 1_000_000
grande = pd.DataFrame({"rover": rng.integers(1, 21, n), "temperatura": rng.normal(-45, 18, n).round(1),
                       "poeira": rng.beta(2, 5, n).round(3)})
grande["temperatura"] += grande.rover * 0.5          # cada rover em um lugar um pouco diferente
ag = grande.groupby("rover").agg(poeira_media=("poeira", "mean"), temp_media=("temperatura", "mean"))
print("mais poeira:", ag.poeira_media.idxmax(), "| mais frios:", list(ag.temp_media.nsmallest(3).index))
grande.to_csv("../saidas/grande.csv", index=False)
print(f"{time.perf_counter() - t0:.1f}s | arquivo: {os.path.getsize('../saidas/grande.csv') / 1e6:.1f} MB")

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
